In [52]:
import pandas as pd
from kneed import KneeLocator
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

🔵 TRAIN AND TEST LASSO REGRESSION ON B2 MICROGLOBULIN

In [53]:
clinical_df = pd.read_csv(r"D:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")
numeric_clinical = clinical_df.select_dtypes(include=['number']).columns

features_df = pd.read_csv(r"D:\CSV\merged\merged_CaSupp_25_radiomics_spine_lesions_features.csv")
numeric_features = features_df.select_dtypes(include=['number']).columns

scaler = StandardScaler()
clinical_scaled = pd.DataFrame(scaler.fit_transform(clinical_df[numeric_clinical]),
                               columns=clinical_df[numeric_clinical].columns,
                               index=clinical_df[numeric_clinical].index)
scaler = StandardScaler()
features_scaled = pd.DataFrame(scaler.fit_transform(features_df[numeric_features]),
                               columns=features_df[numeric_features].columns,
                               index=features_df[numeric_features].index)

valid_idx = clinical_scaled['Beta2 microglobulin (mg/l)'].notna()
lasso = Lasso(alpha=0.01, random_state=42)

X = features_scaled[numeric_features][valid_idx]
y = clinical_scaled['Beta2 microglobulin (mg/l)'][valid_idx]

lasso.fit(X, y)
y_pred = lasso.predict(X)

r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
mae = mean_absolute_error(y, y_pred)

print(f"R²: {r2:.3f} || MSE: {mse:.3f} || MAE: {mae:.3f}")
feature_weights_df = pd.DataFrame({'Feature': X.columns, 'Weight': lasso.coef_}).sort_values(by='Weight', ascending=False).reset_index(drop=True)
feature_weights_df

R²: 0.772 || MSE: 0.228 || MAE: 0.363


,Feature,Weight
0,original_shape_MajorAxisLength,0.939938
1,original_shape_SurfaceVolumeRatio,0.448924
2,gradient_gldm_GrayLevelNonUniformity,0.405301
3,gradient_gldm_LargeDependenceHighGrayLevelEmph...,0.284570
4,original_glcm_MCC,0.248328
...,...,...
196,original_glszm_ZonePercentage,-0.449097
197,original_shape_Maximum3DDiameter,-0.461047
198,n_lesions,-0.478532
199,gradient_ngtdm_Coarseness,-0.562481


🔵 LASSO FOR EVERY DATASET

In [54]:
clinical_df = pd.read_csv(r"D:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")
numeric_clinical = clinical_df.select_dtypes(include=['number']).columns

scaler = StandardScaler()
clinical_scaled = pd.DataFrame(scaler.fit_transform(clinical_df[numeric_clinical]),
                               columns=clinical_df[numeric_clinical].columns,
                               index=clinical_df[numeric_clinical].index)

df_c25 = pd.read_csv(r"D:\CSV\merged\merged_CaSupp_25_radiomics_spine_lesions_features.csv")
df_vmi40 = pd.read_csv(r"D:\CSV\merged\merged_monoe_40kev_radiomics_spine_lesions_features.csv")
df_vmi80 = pd.read_csv(r"D:\CSV\merged\merged_monoe_80kev_radiomics_spine_lesions_features.csv")
df_vmi120 = pd.read_csv(r"D:\CSV\merged\merged_monoe_120kev_radiomics_spine_lesions_features.csv")
df_konv = pd.read_csv(r"D:\CSV\merged\merged_konv_radiomics_spine_lesions_features.csv")

dfs = [df_c25, df_konv, df_vmi40, df_vmi80, df_vmi120]
dataset_names = ['CaSupp_25', 'Konv', 'VMI_40', 'VMI_80', 'VMI_120']

for df, name in zip(dfs, dataset_names):
    numeric_features = df.select_dtypes(include=['number']).columns

    scaler = StandardScaler()
    features_scaled = pd.DataFrame(scaler.fit_transform(features_df[numeric_features]),
                                   columns=features_df[numeric_features].columns,
                                   index=features_df[numeric_features].index)

    valid_idx = clinical_scaled['Beta2 microglobulin (mg/l)'].notna()
    lasso = Lasso(alpha=0.01, random_state=42)

    X = features_scaled[numeric_features][valid_idx]
    y = clinical_scaled['Beta2 microglobulin (mg/l)'][valid_idx]

    lasso.fit(X, y)
    y_pred = lasso.predict(X)

    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    mae = mean_absolute_error(y, y_pred)

    print(name)
    print(f"R²: {r2:.3f} || MSE: {mse:.3f} || MAE: {mae:.3f}")
    feature_weights_df = pd.DataFrame({'Feature': X.columns, 'Weight': lasso.coef_}).sort_values(by='Weight', ascending=False).reset_index(drop=True)
    print(feature_weights_df)

CaSupp_25
R²: 0.772 || MSE: 0.228 || MAE: 0.363
                                               Feature    Weight
0                       original_shape_MajorAxisLength  0.939938
1                    original_shape_SurfaceVolumeRatio  0.448924
2                 gradient_gldm_GrayLevelNonUniformity  0.405301
3    gradient_gldm_LargeDependenceHighGrayLevelEmph...  0.284570
4                                    original_glcm_MCC  0.248328
..                                                 ...       ...
196                      original_glszm_ZonePercentage -0.449097
197                   original_shape_Maximum3DDiameter -0.461047
198                                          n_lesions -0.478532
199                          gradient_ngtdm_Coarseness -0.562481
200                                 gradient_glcm_Imc1 -0.766993

[201 rows x 2 columns]
Konv
R²: 0.772 || MSE: 0.228 || MAE: 0.363
                                               Feature    Weight
0                       original_shape_M

🔵 LASSO FOR EVERY DATASET AND JUST SIGNIFICANT FEATURES

In [55]:
clinical_df = pd.read_csv(r"D:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")
numeric_clinical = clinical_df.select_dtypes(include=['number']).columns

scaler = StandardScaler()
clinical_scaled = pd.DataFrame(scaler.fit_transform(clinical_df[numeric_clinical]),
                               columns=clinical_df[numeric_clinical].columns,
                               index=clinical_df[numeric_clinical].index)

df_c25 = pd.read_csv(r"D:\CSV\merged\merged_CaSupp_25_radiomics_spine_lesions_features.csv")
df_vmi40 = pd.read_csv(r"D:\CSV\merged\merged_monoe_40kev_radiomics_spine_lesions_features.csv")
df_vmi80 = pd.read_csv(r"D:\CSV\merged\merged_monoe_80kev_radiomics_spine_lesions_features.csv")
df_vmi120 = pd.read_csv(r"D:\CSV\merged\merged_monoe_120kev_radiomics_spine_lesions_features.csv")
df_konv = pd.read_csv(r"D:\CSV\merged\merged_konv_radiomics_spine_lesions_features.csv")

sp_c25 = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_CaSupp_25.csv")
sp_konv = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_konv.csv")
sp_vmi40 = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_vmi40.csv")
sp_vmi80 = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_vmi80.csv")
sp_vmi120 = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_vmi120.csv")

significant_c25 = sp_c25[sp_c25["p_value"] < 0.05]['Feature']
significant_konv = sp_konv[sp_konv["p_value"] < 0.05]['Feature']
significant_vmi40 = sp_vmi40[sp_vmi40["p_value"] < 0.05]['Feature']
significant_vmi80 = sp_vmi80[sp_vmi80["p_value"] < 0.05]['Feature']
significant_vmi120 = sp_vmi120[sp_vmi120["p_value"] < 0.05]['Feature']

dfs = [df_c25, df_konv, df_vmi40, df_vmi80, df_vmi120]
dataset_names = ['CaSupp_25', 'Konv', 'VMI_40', 'VMI_80', 'VMI_120']
significant = [significant_c25, significant_konv, significant_vmi40, significant_vmi80, significant_vmi120]

for df, name, significant_features in zip(dfs, dataset_names, significant):

    scaler = StandardScaler()
    features_scaled = pd.DataFrame(scaler.fit_transform(df[significant_features]),
                                   columns=df[significant_features].columns,
                                   index=df[significant_features].index)

    valid_idx = clinical_scaled['Beta2 microglobulin (mg/l)'].notna()
    lasso = Lasso(alpha=0.01, random_state=42)

    X = features_scaled[significant_features][valid_idx]
    y = clinical_scaled['Beta2 microglobulin (mg/l)'][valid_idx]

    lasso.fit(X, y)
    y_pred = lasso.predict(X)

    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    mae = mean_absolute_error(y, y_pred)

    print(name)
    print(f"R²: {r2:.3f} || MSE: {mse:.3f} || MAE: {mae:.3f}")
    feature_weights_df = pd.DataFrame({'Feature': X.columns, 'Weight': lasso.coef_}).sort_values(by='Weight', ascending=False).reset_index(drop=True)
    print(feature_weights_df)

CaSupp_25
R²: 0.346 || MSE: 0.654 || MAE: 0.566
                                            Feature    Weight
0                  gradient_firstorder_90Percentile  0.361174
1                  gradient_gldm_DependenceVariance  0.321074
2         original_firstorder_MeanAbsoluteDeviation  0.247478
3   original_glszm_GrayLevelNonUniformityNormalized  0.214876
4                      original_firstorder_Kurtosis  0.179818
..                                              ...       ...
95           gradient_glrlm_LowGrayLevelRunEmphasis -0.276104
96                       original_firstorder_Median -0.312193
97                       original_glszm_ZoneEntropy -0.339129
98                               gradient_glcm_Imc1 -0.346759
99           gradient_firstorder_InterquartileRange -0.565485

[100 rows x 2 columns]
Konv
R²: 0.422 || MSE: 0.578 || MAE: 0.514
                                               Feature    Weight
0               original_firstorder_InterquartileRange  0.706189
1          